<a href="https://colab.research.google.com/github/Trendtek254/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
agents_url ="https://raw.githubusercontent.com/Trendtek254/plp-da-a3-duka-sales-and-agent-cleanup/refs/heads/main/agents.csv"
transactions_url ="https://raw.githubusercontent.com/Trendtek254/plp-da-a3-duka-sales-and-agent-cleanup/refs/heads/main/agent_transactions.csv"
agents = pd.read_csv(agents_url)
transactions = pd.read_csv(transactions_url)
transactions.head()
print("Agents:",agents.shape)
print("Transactions:",transactions.shape)

# 1. Check missing amounts
missing_amount = transactions["amount"].isna()

# 2. Check exact duplicate records
exact_duplicates = transactions.duplicated()

# 3. Check repeated transaction IDs
duplicate_ids = transactions.duplicated(
    subset="txn_id",
    keep=False
)

# 4. Check transaction IDs with conflicting agent IDs
conflicting_ids = (
    transactions.groupby("txn_id")["agent_id"]
    .transform("nunique") > 1
)

# 5. Check inconsistent transaction types
expected_types = ["Deposit", "Withdrawal"]

inconsistent_types = (
    transactions["txn_type"] !=
    transactions["txn_type"].str.strip().str.title()
)

# 6. Check inconsistent date formats
inconsistent_dates = transactions["txn_date"].str.contains(
    "/", na=False
)

# 7. Check agent IDs missing from the agent register
invalid_agents = ~transactions["agent_id"].isin(agents["agent_id"])

# 8. Flag unusually high amounts for investigation
potential_outliers = transactions["amount"] == 250000

# 9. Create the audit table
audit = pd.DataFrame([
    {
        "issue": "Missing transaction amounts",
        "dimension": "Completeness",
        "affected_column": "amount",
        "affected_rows": missing_amount.sum()
    },
    {
        "issue": "Exact duplicate records",
        "dimension": "Uniqueness",
        "affected_column": "All columns",
        "affected_rows": exact_duplicates.sum()
    },
    {
        "issue": "Repeated transaction IDs",
        "dimension": "Uniqueness",
        "affected_column": "txn_id",
        "affected_rows": duplicate_ids.sum()
    },
    {
        "issue": "Conflicting agent IDs under same transaction ID",
        "dimension": "Uniqueness / Consistency",
        "affected_column": "txn_id, agent_id",
        "affected_rows": conflicting_ids.sum()
    },
    {
        "issue": "Inconsistent transaction type formatting",
        "dimension": "Consistency",
        "affected_column": "txn_type",
        "affected_rows": inconsistent_types.sum()
    },
    {
        "issue": "Inconsistent date formatting",
        "dimension": "Consistency",
        "affected_column": "txn_date",
        "affected_rows": inconsistent_dates.sum()
    },
    {
        "issue": "Agent IDs missing from agent register",
        "dimension": "Referential integrity",
        "affected_column": "agent_id",
        "affected_rows": invalid_agents.sum()
    },
    {
        "issue": "Potential unusually high transaction amounts",
        "dimension": "Validity",
        "affected_column": "amount",
        "affected_rows": potential_outliers.sum()
    }
])

# Display audit table
display(audit)

Agents: (20, 3)
Transactions: (612, 5)


,issue,dimension,affected_column,affected_rows
0,Missing transaction amounts,Completeness,amount,22
1,Exact duplicate records,Uniqueness,All columns,11
2,Repeated transaction IDs,Uniqueness,txn_id,24
3,Conflicting agent IDs under same transaction ID,Uniqueness / Consistency,"txn_id, agent_id",2
4,Inconsistent transaction type formatting,Consistency,txn_type,384
5,Inconsistent date formatting,Consistency,txn_date,5
6,Agent IDs missing from agent register,Referential integrity,agent_id,8
7,Potential unusually high transaction amounts,Validity,amount,5


In [ ]:
audit.to_csv("data_quality_audit.csv",index=False)

***Task11***

In [ ]:
# Count exact duplicate rows
duplicate_count = transactions.duplicated().sum()

print("Exact duplicate transactions:", duplicate_count)



# Preserve the original
txns_raw = transactions.copy()

# Remove exact duplicate rows
txns_clean = transactions.drop_duplicates().copy()

# Calculate number removed
removed = len(transactions) - len(txns_clean)

print("Records before:", len(transactions))
print("Records after:", len(txns_clean))
print("Duplicates removed:", removed)

Exact duplicate transactions: 11
Records before: 612
Records after: 601
Duplicates removed: 11


***Task12***

In [ ]:
# Standardise text temporarily for checking
normalised_type = (
    txns_clean["txn_type"]
    .str.strip()
    .str.title()
)

# Define the only two acceptable values
valid_types = ["Deposit", "Withdrawal"]

# Count missing values
missing_types = txns_clean["txn_type"].isna().sum()

# Count non-missing values that are not valid
invalid_types = (
    normalised_type.notna() &
    ~normalised_type.isin(valid_types)
).sum()

print("Missing transaction types:", missing_types)
print("Invalid transaction types:", invalid_types)


txns_clean["txn_type"] = (
    txns_clean["txn_type"]
    .str.strip()
    .str.title()
)



display(txns_clean["txn_type"].value_counts(dropna=False))

print("Unique transaction types:")
print(txns_clean["txn_type"].unique())





Missing transaction types: 0
Invalid transaction types: 0


,count
txn_type,
Deposit,374
Withdrawal,227


Unique transaction types:
['Deposit' 'Withdrawal']


***Task13***

In [ ]:
# Convert amount from text to numeric
txns_clean["amount"] = pd.to_numeric(
    txns_clean["amount"],
    errors="coerce"
)

# Count missing or invalid amounts
print("Missing or invalid amounts:",
      txns_clean["amount"].isna().sum())

# Check the data type
print("Amount data type:", txns_clean["amount"].dtype)

# Replace forward slashes with hyphens
standard_dates = txns_clean["txn_date"].str.replace(
    "/", "-", regex=False
)

# Convert to datetime
txns_clean["txn_date"] = pd.to_datetime(
    standard_dates,
    format="%Y-%m-%d",
    errors="coerce"
)

# Check the results
print("Missing or invalid dates:",
      txns_clean["txn_date"].isna().sum())

print("Date data type:", txns_clean["txn_date"].dtype)

display(txns_clean[["txn_date"]].head())

display(
    txns_clean[txns_clean["amount"] == 250000]
)

print("Final data types:")
print(txns_clean[["amount", "txn_date"]].dtypes)

print("\nMissing amounts:")
print(txns_clean["amount"].isna().sum())

print("\nMissing dates:")
print(txns_clean["txn_date"].isna().sum())









Missing or invalid amounts: 20
Amount data type: float64
Missing or invalid dates: 0
Date data type: datetime64[ns]


,txn_date
0,2026-03-03
1,2026-04-27
2,2026-04-29
3,2026-04-24
4,2026-04-17


,txn_id,agent_id,txn_type,amount,txn_date
306,TX00307,AG012,Deposit,250000.0,2026-03-02
315,TX00316,AG014,Deposit,250000.0,2026-04-01
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Deposit,250000.0,2026-03-11


Final data types:
amount             float64
txn_date    datetime64[ns]
dtype: object

Missing amounts:
20

Missing dates:
0


***Task14***

In [ ]:
# Reload original CSV without automatically interpreting "N/A" as missing
txns_source = pd.read_csv(
    transactions_url,
    keep_default_na=False
)

# Match original amounts to the rows remaining in txns_clean
raw_amount = (
    txns_source.loc[txns_clean.index, "amount"]
    .astype("string")
    .str.strip()
)

import numpy as np

# Truly blank amounts
missing_mask = raw_amount.eq("")

# Values that cannot be converted to numbers
invalid_mask = (
    ~missing_mask &
    pd.to_numeric(raw_amount, errors="coerce").isna()
)

print("Missing amounts:", missing_mask.sum())
print("Invalid amount entries:", invalid_mask.sum())

# Convert amounts into numeric values
txns_clean["amount"] = pd.to_numeric(
    raw_amount,
    errors="coerce"
)

# Create amount flag with conditions converted to standard boolean arrays
txns_clean["amount_flag"] = np.select(
    [
        missing_mask.astype(bool),
        invalid_mask.astype(bool),
        txns_clean["amount"].eq(250000).fillna(False).astype(bool)
    ],
    [
        "Missing",
        "Invalid",
        "Review"
    ],
    default="Valid"

)


print("Amount flag summary:")
display(txns_clean["amount_flag"].value_counts())

# Inspect problematic transactions
display(
    txns_clean[
        txns_clean["amount_flag"] != "Valid"
    ][["txn_id", "amount", "amount_flag"]]
)



Missing amounts: 15
Invalid amount entries: 5
Amount flag summary:


,count
amount_flag,
Valid,576
Missing,15
Invalid,5
Review,5


,txn_id,amount,amount_flag
10,TX00011,<NA>,Missing
14,TX00015,<NA>,Missing
54,TX00055,<NA>,Missing
66,TX00067,<NA>,Invalid
122,TX00123,<NA>,Missing
140,TX00141,<NA>,Missing
173,TX00174,<NA>,Missing
182,TX00183,<NA>,Missing
192,TX00193,<NA>,Invalid
198,TX00199,<NA>,Missing


***### Completeness: Treatment of Missing and Invalid Amounts

Missing and invalid transaction amounts were retained as missing numeric values (`NaN`) rather than replaced with zero or estimated values. Replacing missing amounts with zero could incorrectly suggest that no money was transacted, while estimating amounts could introduce inaccurate financial information.

A new column, `amount_flag`, was added to identify records as `Missing`, `Invalid`, `Valid`, or `Review`. Transactions with amounts of 250,000 were flagged for further investigation because they are unusually high, but they were not automatically classified as invalid.

This approach preserves the original transaction records, supports transparency and allows questionable transactions to be investigated before analysis.***

***Task 15***

In [ ]:
# Use available numeric amounts only
amounts = txns_clean["amount"].dropna()

# Calculate quartiles
Q1 = amounts.quantile(0.25)
Q3 = amounts.quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower boundary:", lower_bound)
print("Upper boundary:", upper_bound)

# Identify potential outliers
outlier_mask = (
    (txns_clean["amount"] < lower_bound) |
    (txns_clean["amount"] > upper_bound)
)

# Create an IQR flag converting conditions to standard boolean arrays
txns_clean["iqr_flag"] = np.select(
    [
        txns_clean["amount"].isna().astype(bool),
        outlier_mask.fillna(False).astype(bool)
    ],
    [
        "Not assessed",
        "Potential outlier"
    ],
    default="Within range"
)

# Count the results
display(txns_clean["iqr_flag"].value_counts())


suspicious = txns_clean[
    txns_clean["amount"] == 250000
]

display(
    suspicious[
        ["txn_id", "agent_id", "txn_type",
         "amount", "txn_date", "amount_flag", "iqr_flag"]
    ]
)

Q1: 1077.0
Q3: 3078.0
IQR: 2001.0
Lower boundary: -1924.5
Upper boundary: 6079.5


,count
iqr_flag,
Within range,541
Potential outlier,40
Not assessed,20


,txn_id,agent_id,txn_type,amount,txn_date,amount_flag,iqr_flag
306,TX00307,AG012,Deposit,250000.0,2026-03-02,Review,Potential outlier
315,TX00316,AG014,Deposit,250000.0,2026-04-01,Review,Potential outlier
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,Review,Potential outlier
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,Review,Potential outlier
594,TX00595,AG016,Deposit,250000.0,2026-03-11,Review,Potential outlier


***### Outlier Detection and Treatment

The Interquartile Range (IQR) method was used to identify suspicious transaction amounts. Q1 was KES 1,077 and Q3 was KES 3,078, giving an IQR of KES 2,001. The resulting upper boundary was KES 6,079.50.

A total of 40 transactions were flagged as potential outliers. Five transactions had an amount of KES 250,000, which is substantially higher than the upper boundary.

These five transactions were not automatically deleted because an outlier does not necessarily represent an error. The business should verify them against original transaction records and supporting financial evidence. Confirmed legitimate transactions should be retained, while verified errors should be corrected using reliable source information.

A separate IQR flag column was created to support further investigation without altering the original transaction amounts.***

***Task16***

In [ ]:
# Check that agent IDs are unique in the register
print("Duplicate agent IDs:",
      agents["agent_id"].duplicated().sum())

# Merge transactions with the agent register
merged = txns_clean.merge(
    agents,
    on="agent_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Merged records:", len(merged))



# Identify transactions without a matching agent
unknown_mask = merged["_merge"] == "left_only"

unknown_count = unknown_mask.sum()

print("Transactions referencing unknown agents:",
      unknown_count)

# Inspect unknown agent transactions
display(
    merged.loc[
        unknown_mask,
        ["txn_id", "agent_id", "txn_type", "amount", "_merge"]
    ]
)

merged["agent_flag"] = np.where(
    merged["_merge"] == "left_only",
    "Unknown agent",
    "Matched"
)

display(merged["agent_flag"].value_counts())



Duplicate agent IDs: 0
Merged records: 601
Transactions referencing unknown agents: 8


,txn_id,agent_id,txn_type,amount,_merge
55,TX00056,AG099,Deposit,8507.0,left_only
68,TX00069,AG099,Withdrawal,1877.0,left_only
77,TX00078,AG099,Deposit,1770.0,left_only
237,TX00238,AG099,Deposit,4944.0,left_only
267,TX00268,AG099,Deposit,3110.0,left_only
444,TX00445,AG099,Deposit,5309.0,left_only
557,TX00558,AG099,Deposit,1639.0,left_only
600,TX00405,AG099,Deposit,1936.0,left_only


,count
agent_flag,
Matched,593
Unknown agent,8


***### Referential Integrity: Merging Transactions with the Agent Register

The transaction dataset was joined to the agent register using `agent_id` through a left join. This preserved all 601 transactions while adding the corresponding agent information, including county and onboarding date.

The merge identified 8 transactions referencing an unknown agent ID, `AG099`, which was not present in the official register.

These transactions were retained and flagged as `Unknown agent` rather than deleted or assigned to an arbitrary county. The business should verify the agent's registration details before updating the agent register or including these transactions in county-level reporting.

This approach preserves transaction records and prevents inaccurate county-level analysis.***

In [ ]:
print("Total transactions:", len(merged))
print("Unknown agents:", (merged["agent_flag"] == "Unknown agent").sum())
print("Matched agents:", (merged["agent_flag"] == "Matched").sum())

Total transactions: 601
Unknown agents: 8
Matched agents: 593


***Task17***

In [ ]:
# Keep transactions with registered agents
analysis_data = merged[
    merged["_merge"] == "both"
].copy()

print("Transactions included:", len(analysis_data))
print("Transactions with missing amounts:",
      analysis_data["amount"].isna().sum())


county_summary = (
    analysis_data
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

# Round average values to 2 decimal places
county_summary["average_value"] = (
    county_summary["average_value"].round(2)
)

display(county_summary)

analysis_data["week_start"] = (
    analysis_data["txn_date"]
    .dt.to_period("W-SUN")
    .dt.start_time
)
weekly_pivot = pd.pivot_table(
    analysis_data,
    index="week_start",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

county_summary.to_csv(
    "county_level_summary.csv",
    index=False
)

weekly_pivot.to_csv(
    "weekly_transaction_value_by_county.csv"
)








Transactions included: 593
Transactions with missing amounts: 20


,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.58
1,Kiambu,Withdrawal,75274.0,27,2895.15
2,Kisumu,Deposit,431091.0,74,6071.7
3,Kisumu,Withdrawal,74598.0,37,2072.17
4,Mombasa,Deposit,141880.0,72,2086.47
5,Mombasa,Withdrawal,368592.0,61,6042.49
6,Nairobi,Deposit,536484.0,118,4706.0
7,Nairobi,Withdrawal,154930.0,62,2671.21
8,Nakuru,Deposit,181337.0,60,3126.5
9,Nakuru,Withdrawal,326818.0,39,8600.47


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru
week_start,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,967.0
2026-03-02,18317.0,287933.0,283893.0,68967.0,56574.0
2026-03-09,265995.0,23186.0,28701.0,51084.0,28730.0
2026-03-16,22054.0,24572.0,23368.0,41331.0,39795.0
2026-03-23,11504.0,27077.0,17050.0,39597.0,17222.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0
2026-04-06,15030.0,15341.0,45316.0,55010.0,276609.0
2026-04-13,12265.0,25863.0,26730.0,54656.0,24376.0
2026-04-20,24507.0,37439.0,16687.0,61634.0,30862.0


***Task18***

In [86]:
%%writefile cleaning_pipeline.py

import pandas as pd
import numpy as np


def clean_agent_data(txns, agents):

    # Make copies so the original datasets remain unchanged
    txns = txns.copy()
    agents = agents.copy()

    cleaning_log = []

    # 1. Remove exact duplicate transactions
    before = len(txns)
    txns = txns.drop_duplicates().copy()
    removed = before - len(txns)

    cleaning_log.append({
        "Step": "Duplicate removal",
        "Action": "Removed exact duplicate rows",
        "Rows affected": removed
    })

    # 2. Standardise transaction types
    txns["txn_type"] = (
        txns["txn_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.capitalize()
    )

    cleaning_log.append({
        "Step": "Transaction types",
        "Action": "Standardised transaction type formatting",
        "Rows affected": len(txns)
    })

    # 3. Clean transaction amounts
    original_amount = txns["amount"].copy()

    txns["amount"] = pd.to_numeric(
        txns["amount"], errors="coerce"
    )

    txns["amount_flag"] = np.select(
        [
            original_amount.isna(),
            original_amount.notna() & txns["amount"].isna(),
            txns["amount"].eq(250000)
        ],
        [
            "Missing",
            "Invalid",
            "Review"
        ],
        default="Valid"
    )

    # 4. Standardise and convert transaction dates
    txns["txn_date"] = (
        txns["txn_date"]
        .astype("string")
        .str.strip()
        .str.replace("/", "-", regex=False)
    )

    txns["txn_date"] = pd.to_datetime(
        txns["txn_date"], errors="coerce"
    )

    txns["date_flag"] = np.where(
        txns["txn_date"].isna(),
        "Invalid",
        "Valid"
    )

    # 5. Flag potential outliers using IQR
    valid_amounts = txns["amount"].dropna()

    if len(valid_amounts) > 0:
        q1 = valid_amounts.quantile(0.25)
        q3 = valid_amounts.quantile(0.75)
        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        txns["iqr_flag"] = np.where(
            txns["amount"].isna(),
            "Not assessed",
            np.where(
                (txns["amount"] < lower_limit) |
                (txns["amount"] > upper_limit),
                "Potential outlier",
                "Normal"
            )
        )
    else:
        txns["iqr_flag"] = "Not assessed"

    # 6. Merge transactions with agent information
    clean_txns = txns.merge(
        agents,
        on="agent_id",
        how="left",
        indicator=True
    )

    clean_txns["agent_flag"] = np.where(
        clean_txns["_merge"] == "both",
        "Matched",
        "Unknown agent"
    )

    clean_txns.drop(columns=["_merge"], inplace=True)

    cleaning_log.append({
        "Step": "Agent matching",
        "Action": "Matched transactions with agent records",
        "Rows affected": int(
            (clean_txns["agent_flag"] == "Unknown agent").sum()
        )
    })

    # 7. Create cleaning log
    cleaning_log = pd.DataFrame(cleaning_log)

    return clean_txns, cleaning_log

Writing cleaning_pipeline.py


In [87]:
import os

print(os.path.exists("cleaning_pipeline.py"))

True


In [88]:
from cleaning_pipeline import clean_agent_data

print("Cleaning function imported successfully!")

Cleaning function imported successfully!


In [91]:
clean_txns, cleaning_log = clean_agent_data(transactions, agents)

print("Cleaning completed successfully!")

display(cleaning_log)

Cleaning completed successfully!


,Step,Action,Rows affected
0,Duplicate removal,Removed exact duplicate rows,11
1,Transaction types,Standardised transaction type formatting,601
2,Agent matching,Matched transactions with agent records,8


In [93]:
print("Original transactions:", len(transactions))
print("Cleaned transactions:", len(clean_txns))
print("Agents:", len(agents))

Original transactions: 612
Cleaned transactions: 601
Agents: 20


In [94]:
print(cleaning_log.to_string(index=False))

             Step                                   Action  Rows affected
Duplicate removal             Removed exact duplicate rows             11
Transaction types Standardised transaction type formatting            601
   Agent matching  Matched transactions with agent records              8


***# Data Quality Report: Mobile Money Agent Network

## 1. Introduction

This report evaluates the quality of the mobile money agent transaction dataset and the agent register. The original transaction dataset contained **612 rows**, while the agent register contained **20 agents**. The cleaning pipeline was used to identify and address data quality issues while retaining records that required further investigation.

## 2. Data Quality Assessment

### 2.1 Uniqueness
The transaction dataset contained **11 exact duplicate rows**. These were identified and removed using duplicate detection, leaving **601 transaction rows**. This reduces the risk of counting the same transaction more than once.

### 2.2 Completeness
Some transaction records contained missing or unavailable amounts. These values were not automatically replaced with zero or an average because doing so could distort financial summaries. Missing values were retained and flagged for further investigation.

### 2.3 Consistency
Transaction type labels appeared in different formats, including variations in capitalisation and unnecessary spaces. These were standardised into a consistent format, such as `Deposit` and `Withdrawal`, to support accurate grouping and reporting.

### 2.4 Validity
Transaction amounts and dates were converted into appropriate data types. Invalid values were identified during conversion, while unusual amounts, including values of 250,000, were retained for review. Potential outliers were flagged using the Interquartile Range (IQR) method rather than automatically deleted.

### 2.5 Accuracy
The transaction records were matched against the agent register using `agent_id`. **Eight transactions** could not be matched to a known agent and were flagged as `Unknown agent`. These records were retained so they could be investigated instead of being silently discarded. Further verification against original transaction records would be needed to establish the accuracy of individual amounts and dates.

### 2.6 Timeliness
Transaction dates were standardised and converted into datetime values to support chronological and weekly analysis. However, the recency of the transactions and whether they were submitted on time would require comparison with the expected reporting period and source records.

## 3. Privacy and Data Protection

If the extract also contained customer phone numbers, these would qualify as personal data under Kenya's Data Protection Act, 2019, because they can identify or help identify individuals. Other identifying fields, such as names, identification numbers and addresses, would also be personal data. Protection measures would include collecting only necessary information, restricting access to authorised personnel, using pseudonymised identifiers during analysis, encrypting stored files, and avoiding disclosure of personal details in reports. Personal data should only be processed for a lawful purpose and retained only as long as necessary.

## 4. Conclusion

The cleaning process reduced the transaction dataset from 612 to 601 rows by removing 11 exact duplicates. It also standardised transaction types, flagged unmatched agents and prepared amounts and dates for analysis. Some issues, such as missing values, unusual amounts and unknown agents, require further investigation. Overall, the cleaned dataset is more suitable for analysis, provided that flagged records and limitations are considered when interpreting the results.***